In [1]:
# Import libraries
import pandas as pd
import numpy as np
import os

# Create output directory if it doesn't exist
os.makedirs('output', exist_ok=True)

# Load wrangled data from Q3
df = pd.read_csv('output/q3_wrangled_data.csv', parse_dates=['Measurement Timestamp'], index_col='Measurement Timestamp')
# Or if you saved without index:
# df = pd.read_csv('output/q3_wrangled_data.csv')
# df['Measurement Timestamp'] = pd.to_datetime(df['Measurement Timestamp'])
# df = df.set_index('Measurement Timestamp')
print(f"Loaded {len(df):,} records with datetime index")

Loaded 195,892 records with datetime index


In [2]:
# ========================================
# STEP 2: IDENTIFY NUMERIC COLUMNS FOR ANALYSIS
# ========================================
print("\n" + "="*60)
print("IDENTIFYING VARIABLES FOR ANALYSIS")
print("="*60)

# Get all numeric columns
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"\nTotal numeric columns: {len(numeric_cols)}")

# Identify key sensor variables (exclude derived/temporal features for clarity)
# You may want to focus on original sensor readings
temporal_features = ['hour', 'day_of_week', 'month', 'year', 'is_weekend', 
                     'day_of_month', 'quarter']

# Key variables for analysis (adjust based on your data)
key_vars = [col for col in numeric_cols if col not in temporal_features 
            and not col.startswith('rolling') 
            and not col.endswith('_squared')
            and not col.endswith('_category')]

print(f"\nKey variables for pattern analysis: {len(key_vars)}")
for var in key_vars[:10]:  # Show first 10
    print(f"  - {var}")


IDENTIFYING VARIABLES FOR ANALYSIS

Total numeric columns: 21

Key variables for pattern analysis: 14
  - Air Temperature
  - Wet Bulb Temperature
  - Humidity
  - Rain Intensity
  - Interval Rain
  - Total Rain
  - Precipitation Type
  - Wind Direction
  - Wind Speed
  - Maximum Wind Speed


In [3]:

# ========================================
# STEP 3: TEMPORAL TREND ANALYSIS
# ========================================
print("\n" + "="*60)
print("STEP 1: TEMPORAL TREND ANALYSIS")
print("="*60)

print("\nAggregating data by time periods...")

# Monthly averages (use 'ME' for month end)
print("\nCalculating monthly averages...")
monthly_avg = df[key_vars].resample('ME').mean()
print(f"  Monthly data points: {len(monthly_avg)}")

# Daily averages
print("Calculating daily averages...")
daily_avg = df[key_vars].resample('D').mean()
print(f"  Daily data points: {len(daily_avg)}")

# Calculate overall trends
print("\nOverall trend analysis:")
for var in key_vars[:5]:  # Analyze first 5 variables
    overall_mean = df[var].mean()
    overall_std = df[var].std()
    overall_min = df[var].min()
    overall_max = df[var].max()
    print(f"\n{var}:")
    print(f"  Mean: {overall_mean:.2f}")
    print(f"  Std: {overall_std:.2f}")
    print(f"  Range: [{overall_min:.2f}, {overall_max:.2f}]")

# ========================================
# STEP 4: SEASONAL PATTERN ANALYSIS
# ========================================
print("\n" + "="*60)
print("STEP 2: SEASONAL PATTERN ANALYSIS")
print("="*60)

# Monthly patterns
print("\nAnalyzing monthly patterns...")
if 'month' in df.columns:
    monthly_pattern = df.groupby('month')[key_vars].mean()
    print("✓ Monthly patterns calculated")
    print("\nSample monthly pattern (first variable):")
    if len(key_vars) > 0:
        print(monthly_pattern[[key_vars[0]]].head())

# Hourly patterns (diurnal cycle)
print("\nAnalyzing hourly patterns (diurnal cycle)...")
if 'hour' in df.columns:
    hourly_pattern = df.groupby('hour')[key_vars].mean()
    print("✓ Hourly patterns calculated")
    print("\nSample hourly pattern (first variable):")
    if len(key_vars) > 0:
        print(hourly_pattern[[key_vars[0]]].head())

# Day of week patterns
print("\nAnalyzing day of week patterns...")
if 'day_of_week' in df.columns:
    dow_pattern = df.groupby('day_of_week')[key_vars].mean()
    print("✓ Day of week patterns calculated")

# Weekend vs weekday patterns
print("\nAnalyzing weekend vs weekday patterns...")
if 'is_weekend' in df.columns:
    weekend_pattern = df.groupby('is_weekend')[key_vars].mean()
    print("✓ Weekend patterns calculated")
    print("\nWeekend effect (first variable):")
    if len(key_vars) > 0:
        weekday_val = weekend_pattern.loc[0, key_vars[0]]
        weekend_val = weekend_pattern.loc[1, key_vars[0]]
        print(f"  Weekday: {weekday_val:.2f}")
        print(f"  Weekend: {weekend_val:.2f}")
        print(f"  Difference: {weekend_val - weekday_val:.2f}")



STEP 1: TEMPORAL TREND ANALYSIS

Aggregating data by time periods...

Calculating monthly averages...
  Monthly data points: 128
Calculating daily averages...
  Daily data points: 3867

Overall trend analysis:

Air Temperature:
  Mean: 12.65
  Std: 10.43
  Range: [-29.78, 37.60]

Wet Bulb Temperature:
  Mean: 10.29
  Std: 9.40
  Range: [-28.90, 28.40]

Humidity:
  Mean: 68.02
  Std: 15.64
  Range: [0.00, 100.00]

Rain Intensity:
  Mean: 0.16
  Std: 1.80
  Range: [0.00, 183.60]

Interval Rain:
  Mean: 0.14
  Std: 1.10
  Range: [-0.90, 63.42]

STEP 2: SEASONAL PATTERN ANALYSIS

Analyzing monthly patterns...
✓ Monthly patterns calculated

Sample monthly pattern (first variable):
       Air Temperature
month                 
1            -2.551031
2             0.501822
3             4.255540
4             8.633969
5            14.835429

Analyzing hourly patterns (diurnal cycle)...
✓ Hourly patterns calculated

Sample hourly pattern (first variable):
      Air Temperature
hour           

In [4]:
# ========================================
# STEP 3: CALCULATE ROLLING WINDOWS
# ========================================
print("\n" + "="*60)
print("STEP 2: CALCULATING ROLLING WINDOW FEATURES")
print("="*60)

# Ensure data is sorted by datetime index (critical for rolling windows!)
print("\nSorting data by datetime index...")
df = df.sort_index()
print("✓ Data sorted chronologically")

print("\nCalculating rolling window features...")
print("Note: Using predictor variables only (NOT the target variable)")

rolling_features = []

# Identify which columns to create rolling features for
# DO NOT include your target variable here!
# For example, if predicting Air Temperature, don't include it in rolling features

# Rolling windows for Wind Speed
if 'Wind Speed' in df.columns:
    print("\nWind Speed rolling windows:")
    
    # 7-hour rolling mean
    df['wind_speed_rolling_7h'] = df['Wind Speed'].rolling(window=7, min_periods=1).mean()
    rolling_features.append('wind_speed_rolling_7h')
    print("  ✓ wind_speed_rolling_7h: 7-hour rolling mean")
    
    # 24-hour rolling mean
    df['wind_speed_rolling_24h'] = df['Wind Speed'].rolling(window=24, min_periods=1).mean()
    rolling_features.append('wind_speed_rolling_24h')
    print("  ✓ wind_speed_rolling_24h: 24-hour rolling mean")
    
    # 7-hour rolling std (volatility)
    df['wind_speed_rolling_std_7h'] = df['Wind Speed'].rolling(window=7, min_periods=1).std()
    rolling_features.append('wind_speed_rolling_std_7h')
    print("  ✓ wind_speed_rolling_std_7h: 7-hour rolling std")

# Rolling windows for Humidity
if 'Humidity' in df.columns:
    print("\nHumidity rolling windows:")
    
    # 7-hour rolling mean
    df['humidity_rolling_7h'] = df['Humidity'].rolling(window=7, min_periods=1).mean()
    rolling_features.append('humidity_rolling_7h')
    print("  ✓ humidity_rolling_7h: 7-hour rolling mean")
    
    # 24-hour rolling mean
    df['humidity_rolling_24h'] = df['Humidity'].rolling(window=24, min_periods=1).mean()
    rolling_features.append('humidity_rolling_24h')
    print("  ✓ humidity_rolling_24h: 24-hour rolling mean")

# Rolling windows for Water Temperature (if it's a predictor, not target)
if 'Water Temperature' in df.columns:
    print("\nWater Temperature rolling windows:")
    
    # 7-hour rolling mean
    df['water_temp_rolling_7h'] = df['Water Temperature'].rolling(window=7, min_periods=1).mean()
    rolling_features.append('water_temp_rolling_7h')
    print("  ✓ water_temp_rolling_7h: 7-hour rolling mean")
    
    # 24-hour rolling mean
    df['water_temp_rolling_24h'] = df['Water Temperature'].rolling(window=24, min_periods=1).mean()
    rolling_features.append('water_temp_rolling_24h')
    print("  ✓ water_temp_rolling_24h: 24-hour rolling mean")

# Rolling windows for Barometric Pressure
if 'Barometric Pressure' in df.columns:
    print("\nBarometric Pressure rolling windows:")
    
    # 7-hour rolling mean
    df['pressure_rolling_7h'] = df['Barometric Pressure'].rolling(window=7, min_periods=1).mean()
    rolling_features.append('pressure_rolling_7h')
    print("  ✓ pressure_rolling_7h: 7-hour rolling mean")
    
    # 24-hour rolling mean
    df['pressure_rolling_24h'] = df['Barometric Pressure'].rolling(window=24, min_periods=1).mean()
    rolling_features.append('pressure_rolling_24h')
    print("  ✓ pressure_rolling_24h: 24-hour rolling mean")

print(f"\nTotal rolling features created: {len(rolling_features)}")

# Add rolling features to new features list
new_features.extend(rolling_features)



STEP 2: CALCULATING ROLLING WINDOW FEATURES

Sorting data by datetime index...
✓ Data sorted chronologically

Calculating rolling window features...
Note: Using predictor variables only (NOT the target variable)

Wind Speed rolling windows:
  ✓ wind_speed_rolling_7h: 7-hour rolling mean
  ✓ wind_speed_rolling_24h: 24-hour rolling mean
  ✓ wind_speed_rolling_std_7h: 7-hour rolling std

Humidity rolling windows:
  ✓ humidity_rolling_7h: 7-hour rolling mean
  ✓ humidity_rolling_24h: 24-hour rolling mean

Barometric Pressure rolling windows:
  ✓ pressure_rolling_7h: 7-hour rolling mean
  ✓ pressure_rolling_24h: 24-hour rolling mean

Total rolling features created: 7


NameError: name 'new_features' is not defined

In [ ]:

# ========================================
# STEP 4: CREATE CATEGORICAL FEATURES
# ========================================
print("\n" + "="*60)
print("STEP 3: CREATING CATEGORICAL FEATURES")
print("="*60)

categorical_features = []

# Temperature categories (if Air Temperature exists)
if 'Air Temperature' in df.columns:
    print("\nTemperature categories:")
    
    df['temp_category'] = pd.cut(df['Air Temperature'], 
                                bins=[-np.inf, 10, 20, 30, np.inf],
                                labels=['Cold', 'Cool', 'Warm', 'Hot'])
    categorical_features.append('temp_category')
    print("  ✓ temp_category: Cold/Cool/Warm/Hot")
    print(f"    Distribution:\n{df['temp_category'].value_counts()}")

# Time of day categories
if 'hour' in df.columns:
    print("\nTime of day categories:")
    
    df['time_of_day'] = pd.cut(df['hour'], 
                                bins=[-1, 6, 12, 18, 24],
                                labels=['Night', 'Morning', 'Afternoon', 'Evening'])
    categorical_features.append('time_of_day')
    print("  ✓ time_of_day: Night/Morning/Afternoon/Evening")
    print(f"    Distribution:\n{df['time_of_day'].value_counts()}")

# Season categories (if month exists)
if 'month' in df.columns:
    print("\nSeason categories:")
    
    df['season'] = pd.cut(df['month'], 
                        bins=[0, 3, 6, 9, 12],
                        labels=['Winter', 'Spring', 'Summer', 'Fall'])
    categorical_features.append('season')
    print("  ✓ season: Winter/Spring/Summer/Fall")
    print(f"    Distribution:\n{df['season'].value_counts()}")

print(f"\nTotal categorical features created: {len(categorical_features)}")

# Add categorical features to new features list
new_features.extend(categorical_features)



STEP 3: CREATING CATEGORICAL FEATURES

Temperature categories:
  ✓ temp_category: Cold/Cool/Warm/Hot
    Distribution:
temp_category
Cold    80347
Warm    59261
Cool    53161
Hot      3048
Name: count, dtype: int64

Time of day categories:
  ✓ time_of_day: Night/Morning/Afternoon/Evening
    Distribution:
time_of_day
Night        56701
Afternoon    49185
Morning      49162
Evening      40844
Name: count, dtype: int64

Season categories:
  ✓ season: Winter/Spring/Summer/Fall
    Distribution:
season
Summer    56730
Spring    49725
Fall      49331
Winter    40106
Name: count, dtype: int64

Total categorical features created: 3


In [ ]:

# ========================================
# STEP 5: HANDLE MISSING VALUES IN NEW FEATURES
# ========================================
print("\n" + "="*60)
print("STEP 4: HANDLING MISSING VALUES IN NEW FEATURES")
print("="*60)

# Check for missing values in new features
print("\nChecking for missing values in new features...")
new_features_df = df[new_features]
missing_in_new = new_features_df.isnull().sum()
missing_any = missing_in_new[missing_in_new > 0]

if len(missing_any) > 0:
    print(f"\nFound missing values in {len(missing_any)} features:")
    for col, count in missing_any.items():
        pct = (count / len(df)) * 100
        print(f"  - {col}: {count} ({pct:.2f}%)")
    
    # Handle missing values (rolling windows often have NaN at the start)
    print("\nHandling missing values...")
    # For rolling features, forward-fill is appropriate
    for col in rolling_features:
        if df[col].isnull().sum() > 0:
            df[col] = df[col].fillna(method='bfill')  # Backward fill for initial values
    print("  ✓ Missing values in rolling features handled")
else:
    print("  ✓ No missing values in new features")


STEP 4: HANDLING MISSING VALUES IN NEW FEATURES

Checking for missing values in new features...

Found missing values in 7 features:
  - comfort_index: 75 (0.04%)
  - temp_wind_interaction: 75 (0.04%)
  - pressure_deviation: 146 (0.07%)
  - wind_speed_rolling_std_7h: 1 (0.00%)
  - pressure_rolling_7h: 140 (0.07%)
  - pressure_rolling_24h: 123 (0.06%)
  - temp_category: 75 (0.04%)

Handling missing values...
  ✓ Missing values in rolling features handled


/var/folders/6w/gv8g6zzd417_9g9fl64d_nhc0000gn/T/ipykernel_39222/3971671773.py:25: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df[col] = df[col].fillna(method='bfill')  # Backward fill for initial values


In [ ]:

# ========================================
# SAVE ARTIFACT 1: q4_features.csv
# ========================================
print("\n" + "="*60)
print("SAVING ARTIFACTS")
print("="*60)

print("\nSaving all features...")
# Reset index to save datetime as a column
df_to_save = df.reset_index()
df_to_save.to_csv('output/q4_features.csv', index=False)
print("✓ Saved: output/q4_features.csv")
print(f"  Columns saved: {df_to_save.shape[1]}")
print(f"  Rows saved: {df_to_save.shape[0]}")

# ========================================
# SAVE ARTIFACT 2: q4_rolling_features.csv
# ========================================
print("\nSaving rolling features...")
# Select only rolling features and reset index
rolling_df = df[rolling_features].reset_index()
rolling_df.to_csv('output/q4_rolling_features.csv', index=False)
print("✓ Saved: output/q4_rolling_features.csv")
print(f"  Columns saved: {rolling_df.shape[1]}")
print(f"  Rolling features: {len(rolling_features)}")

# Display sample
print("\nSample of rolling features:")
print(rolling_df.head())

# ========================================
# SAVE ARTIFACT 3: q4_feature_list.txt
# ========================================
print("\nSaving feature list...")
with open('output/q4_feature_list.txt', 'w') as f:
    for feature in new_features:
        f.write(f"{feature}\n")

print("✓ Saved: output/q4_feature_list.txt")
print(f"  Total features listed: {len(new_features)}")


SAVING ARTIFACTS

Saving all features...
✓ Saved: output/q4_features.csv
  Columns saved: 42
  Rows saved: 195892

Saving rolling features...
✓ Saved: output/q4_rolling_features.csv
  Columns saved: 8
  Rolling features: 7

Sample of rolling features:
  Measurement Timestamp  wind_speed_rolling_7h  wind_speed_rolling_24h  \
0   2015-04-25 09:00:00               5.100000                5.100000   
1   2015-04-30 05:00:00               6.150000                6.150000   
2   2015-05-22 15:00:00               4.733333                4.733333   
3   2015-05-22 16:00:00               4.550000                4.550000   
4   2015-05-22 17:00:00               3.880000                3.880000   

   wind_speed_rolling_std_7h  humidity_rolling_7h  humidity_rolling_24h  \
0                   1.484924            86.000000             86.000000   
1                   1.484924            81.000000             81.000000   
2                   2.668957            72.333333             72.333333   
3 

In [ ]:

# ========================================
# VERIFICATION
# ========================================
print("\n" + "="*60)
print("VERIFICATION")
print("="*60)

print("\nVerifying feature creation:")
print(f"  Derived features: {len(new_features) - len(rolling_features) - len(categorical_features)}")
print(f"  Rolling features: {len(rolling_features)}")
print(f"  Categorical features: {len(categorical_features)}")
print(f"  Total new features: {len(new_features)}")

print("\nNew features created:")
for i, feature in enumerate(new_features, 1):
    print(f"  {i}. {feature}")

# Verify files exist
print("\nVerifying output files:")
output_files = [
    'output/q4_features.csv',
    'output/q4_rolling_features.csv',
    'output/q4_feature_list.txt'
]
for file in output_files:
    if os.path.exists(file):
        size = os.path.getsize(file) / 1024  # KB
        print(f"  ✓ {file} ({size:.1f} KB)")
    else:
        print(f"  ✗ {file} NOT FOUND!")

# Check for any remaining issues
print("\nFinal data quality check:")
total_missing = df.isnull().sum().sum()
print(f"  Total missing values: {total_missing}")
print(f"  Total rows: {len(df):,}")
print(f"  Total columns: {df.shape[1]}")

# ========================================
# SUMMARY
# ========================================
print("\n" + "="*60)
print("Q4 COMPLETE - All artifacts created successfully!")
print("="*60)
print("\nFiles created:")
print("  1. output/q4_features.csv")
print("  2. output/q4_rolling_features.csv")
print("  3. output/q4_feature_list.txt")
print("\nFeature Engineering Summary:")
print(f"  - Original columns: {df.shape[1] - len(new_features)}")
print(f"  - New features: {len(new_features)}")
print(f"  - Total columns: {df.shape[1]}")
print(f"  - Derived features: {len(new_features) - len(rolling_features) - len(categorical_features)}")
print(f"  - Rolling features: {len(rolling_features)}")
print(f"  - Categorical features: {len(categorical_features)}")
print("\nNext: Proceed to Q5 for pattern analysis")
print("="*60)


VERIFICATION

Verifying feature creation:
  Derived features: 6
  Rolling features: 7
  Categorical features: 3
  Total new features: 16

New features created:
  1. wind_speed_squared
  2. wind_category
  3. comfort_index
  4. humidity_squared
  5. temp_wind_interaction
  6. pressure_deviation
  7. wind_speed_rolling_7h
  8. wind_speed_rolling_24h
  9. wind_speed_rolling_std_7h
  10. humidity_rolling_7h
  11. humidity_rolling_24h
  12. pressure_rolling_7h
  13. pressure_rolling_24h
  14. temp_category
  15. time_of_day
  16. season

Verifying output files:
  ✓ output/q4_features.csv (72884.5 KB)
  ✓ output/q4_rolling_features.csv (26430.3 KB)
  ✓ output/q4_feature_list.txt (0.3 KB)

Final data quality check:
  Total missing values: 379272
  Total rows: 195,892
  Total columns: 41

Q4 COMPLETE - All artifacts created successfully!

Files created:
  1. output/q4_features.csv
  2. output/q4_rolling_features.csv
  3. output/q4_feature_list.txt

Feature Engineering Summary:
  - Original co